<a href="https://colab.research.google.com/github/robinsonvelag-ops/Unicafam_Modelos_Analiticos_Indra/blob/main/Sesion3_Integracion_de_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Integración de datos**

In [26]:
import pandas as pd
import numpy as np
from io import StringIO

In [27]:
csv_clientes = """id_cliente;nombre;ciudad;segmento
C001;Ana Ruiz;Bogotá;Oro
C002;Luis Peña;Medellín;Plata
C003;Marta Gil;Cali;Oro
C004;Juan Ríos;Bogotá;Bronce
C005;Sofía Mora;Barranquilla;Plata"""

In [28]:
df_california_housing = pd.read_csv("/content/sample_data/california_housing_train.csv")

In [29]:
df_california_housing

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0
...,...,...,...,...,...,...,...,...,...
16995,-124.26,40.58,52.0,2217.0,394.0,907.0,369.0,2.3571,111400.0
16996,-124.27,40.69,36.0,2349.0,528.0,1194.0,465.0,2.5179,79000.0
16997,-124.30,41.84,17.0,2677.0,531.0,1244.0,456.0,3.0313,103600.0
16998,-124.30,41.80,19.0,2672.0,552.0,1298.0,478.0,1.9797,85800.0


In [30]:
clientes = pd.read_csv(StringIO(csv_clientes), sep=";")

In [31]:
clientes

,id_cliente,nombre,ciudad,segmento
0,C001,Ana Ruiz,Bogotá,Oro
1,C002,Luis Peña,Medellín,Plata
2,C003,Marta Gil,Cali,Oro
3,C004,Juan Ríos,Bogotá,Bronce
4,C005,Sofía Mora,Barranquilla,Plata


In [32]:
print(clientes.shape)
print(clientes.head(3))

(5, 4)
  id_cliente     nombre    ciudad segmento
0       C001   Ana Ruiz    Bogotá      Oro
1       C002  Luis Peña  Medellín    Plata
2       C003  Marta Gil      Cali      Oro


In [33]:
df_california_housing.shape

(17000, 9)

In [34]:
df_california_housing.head(4)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0


In [35]:
#Integrar un archivo JSON o  via API
respuesta_api = [
    {"pedido": 101, "cliente": "C001", "items": 3,
     "pago": {"medio": "Tarjeta", "valor": 150000}},
    {"pedido": 102, "cliente": "C003", "items": 1,
     "pago": {"medio": "PSE", "valor": 80000}},
    {"pedido": 103, "cliente": "C001", "items": 2,
     "pago": {"medio": "Efectivo", "valor": 45000}},
    {"pedido": 104, "cliente": "C009", "items": 5,
     "pago": {"medio": "Tarjeta", "valor": 210000}},
]
pedidos = pd.json_normalize(respuesta_api)
print(pedidos)



   pedido cliente  items pago.medio  pago.valor
0     101    C001      3    Tarjeta      150000
1     102    C003      1        PSE       80000
2     103    C001      2   Efectivo       45000
3     104    C009      5    Tarjeta      210000


# **Merge de datos**

In [36]:
#merge: INNER vs LEFT
inner = pd.merge(clientes, pedidos, left_on="id_cliente",
                 right_on="cliente", how="inner")
left = pd.merge(clientes, pedidos, left_on="id_cliente",
                right_on="cliente", how="left")

print("INNER:", inner.shape[0], "filas")
print("LEFT :", left.shape[0], "filas")
sin_compra = left[left["pedido"].isna()]["nombre"].tolist()
print("Clientes sin compras:", sin_compra)


INNER: 3 filas
LEFT : 6 filas
Clientes sin compras: ['Luis Peña', 'Juan Ríos', 'Sofía Mora']


In [37]:
#Validar con indicator
outer = pd.merge(clientes, pedidos, left_on="id_cliente",
                 right_on="cliente", how="outer", indicator=True)
print(outer["_merge"].value_counts())
huerfanos = outer[outer["_merge"] == "right_only"]
print("Pedidos sin cliente:", huerfanos["pedido"].tolist())


_merge
left_only     3
both          3
right_only    1
Name: count, dtype: int64
Pedidos sin cliente: [104.0]


# **Consolidación de datos**

In [38]:
np.random.seed(1)

def ventas_mes(mes):
    return pd.DataFrame({
        "mes": mes,
        "sucursal": ["Norte", "Sur", "Centro"],
        "ventas": np.random.randint(80, 200, 3) * 1e6})

In [39]:
ene = ventas_mes("Ene")
ene

,mes,sucursal,ventas
0,Ene,Norte,117000000.0
1,Ene,Sur,187000000.0
2,Ene,Centro,92000000.0


In [40]:
feb = ventas_mes("Feb")
feb

,mes,sucursal,ventas
0,Feb,Norte,152000000.0
1,Feb,Sur,89000000.0
2,Feb,Centro,155000000.0


In [41]:
mar = ventas_mes("Mar")
mar

,mes,sucursal,ventas
0,Mar,Norte,85000000.0
1,Mar,Sur,159000000.0
2,Mar,Centro,144000000.0


In [42]:
consolidado = pd.concat([ene, feb, mar], ignore_index=True)

print(consolidado.shape)

print(consolidado.tail(3))

(9, 3)
   mes sucursal       ventas
6  Mar    Norte   85000000.0
7  Mar      Sur  159000000.0
8  Mar   Centro  144000000.0


In [43]:
consolidado

,mes,sucursal,ventas
0,Ene,Norte,117000000.0
1,Ene,Sur,187000000.0
2,Ene,Centro,92000000.0
3,Feb,Norte,152000000.0
4,Feb,Sur,89000000.0
5,Feb,Centro,155000000.0
6,Mar,Norte,85000000.0
7,Mar,Sur,159000000.0
8,Mar,Centro,144000000.0


In [44]:
# Group By con varias metricas
resumen = consolidado.groupby("sucursal").agg(
    total=("ventas", "sum"),
    promedio=("ventas", "mean"),
    meses=("mes", "count"))
resumen = (resumen / [1e6, 1e6, 1]).round(1)
print(resumen.sort_values("total", ascending=False))


          total  promedio  meses
sucursal                        
Sur       435.0     145.0    3.0
Centro    391.0     130.3    3.0
Norte     354.0     118.0    3.0


In [45]:
# Pivot y Melt
ancho = consolidado.pivot_table(index="sucursal", columns="mes",
                                values="ventas") / 1e6
ancho = ancho[["Ene", "Feb", "Mar"]]
print(ancho)

largo = ancho.reset_index().melt(id_vars="sucursal",
                                 var_name="mes", value_name="ventas_M")
print(largo.head(4))


mes         Ene    Feb    Mar
sucursal                     
Centro     92.0  155.0  144.0
Norte     117.0  152.0   85.0
Sur       187.0   89.0  159.0
  sucursal  mes  ventas_M
0   Centro  Ene      92.0
1    Norte  Ene     117.0
2      Sur  Ene     187.0
3   Centro  Feb     155.0


# **Integración SQL**

In [46]:
import sqlite3
con = sqlite3.connect(":memory:")
clientes.to_sql("clientes", con, index=False)
pedidos.rename(columns={"pago.valor": "valor"})[
    ["pedido", "cliente", "valor"]].to_sql("pedidos", con, index=False)

query = """
SELECT c.ciudad, COUNT(p.pedido) AS pedidos, SUM(p.valor) AS ventas
FROM clientes c LEFT JOIN pedidos p ON c.id_cliente = p.cliente
GROUP BY c.ciudad ORDER BY ventas DESC
"""
print(pd.read_sql(query, con))


         ciudad  pedidos    ventas
0        Bogotá        2  195000.0
1          Cali        1   80000.0
2      Medellín        0       NaN
3  Barranquilla        0       NaN


# **EJERCICIO**

## **Ejercicio 1: Clientes inactivos**
Marketing quiere enviar una campaña a los clientes que nunca han comprado y revisar los pedidos con clientes inexistentes.

In [47]:
# Agrega 3 clientes nuevos (C006–C008) a la tabla clientes
nuevos = pd.DataFrame({
    "id_cliente": ["C006", "C007", "C008"],
    "nombre": ["Diana Díaz", "Ana Soto", "Adriana Cruz"],
    "ciudad": ["Bogotá", "Medellín", "Cali"]
})

clientes = pd.concat([clientes, nuevos], ignore_index=True)

clientes

,id_cliente,nombre,ciudad,segmento
0,C001,Ana Ruiz,Bogotá,Oro
1,C002,Luis Peña,Medellín,Plata
2,C003,Marta Gil,Cali,Oro
3,C004,Juan Ríos,Bogotá,Bronce
4,C005,Sofía Mora,Barranquilla,Plata
5,C006,Diana Díaz,Bogotá,NaN
6,C007,Ana Soto,Medellín,NaN
7,C008,Adriana Cruz,Cali,NaN


In [50]:
# Haz un LEFT JOIN con pedidos
left2 = pd.merge(clientes, pedidos, left_on="id_cliente",
                 right_on="cliente", how="left")

# Lista los clientes sin compras.
left2[left2["cliente"].isna()]

,id_cliente,nombre,ciudad,segmento,pedido,cliente,items,pago.medio,pago.valor
2,C002,Luis Peña,Medellín,Plata,NaN,NaN,NaN,NaN,NaN
4,C004,Juan Ríos,Bogotá,Bronce,NaN,NaN,NaN,NaN,NaN
5,C005,Sofía Mora,Barranquilla,Plata,NaN,NaN,NaN,NaN,NaN
6,C006,Diana Díaz,Bogotá,NaN,NaN,NaN,NaN,NaN,NaN
7,C007,Ana Soto,Medellín,NaN,NaN,NaN,NaN,NaN,NaN
8,C008,Adriana Cruz,Cali,NaN,NaN,NaN,NaN,NaN,NaN


In [53]:
# Haz un OUTER JOIN con indicator=True
outer2 = pd.merge(clientes, pedidos, left_on="id_cliente",
                    right_on="cliente", how="outer", indicator=True)

outer2

,id_cliente,nombre,ciudad,segmento,pedido,cliente,items,pago.medio,pago.valor,_merge
0,C001,Ana Ruiz,Bogotá,Oro,101.0,C001,3.0,Tarjeta,150000.0,both
1,C001,Ana Ruiz,Bogotá,Oro,103.0,C001,2.0,Efectivo,45000.0,both
2,C002,Luis Peña,Medellín,Plata,NaN,NaN,NaN,NaN,NaN,left_only
3,C003,Marta Gil,Cali,Oro,102.0,C003,1.0,PSE,80000.0,both
4,C004,Juan Ríos,Bogotá,Bronce,NaN,NaN,NaN,NaN,NaN,left_only
5,C005,Sofía Mora,Barranquilla,Plata,NaN,NaN,NaN,NaN,NaN,left_only
6,C006,Diana Díaz,Bogotá,NaN,NaN,NaN,NaN,NaN,NaN,left_only
7,C007,Ana Soto,Medellín,NaN,NaN,NaN,NaN,NaN,NaN,left_only
8,C008,Adriana Cruz,Cali,NaN,NaN,NaN,NaN,NaN,NaN,left_only
9,NaN,NaN,NaN,NaN,104.0,C009,5.0,Tarjeta,210000.0,right_only


In [54]:
# Cuenta cada categoría de _merge.
outer2["_merge"].value_counts()

,count
_merge,
left_only,6
both,3
right_only,1


In [55]:
# Escribe la lista final para marketing
clientes_inactivos = outer2[outer2["_merge"] == "left_only"]

print("Clientes para la campaña de marketing:")
print(clientes_inactivos[["id_cliente", "nombre", "ciudad"]])

Clientes para la campaña de marketing:
  id_cliente        nombre        ciudad
2       C002     Luis Peña      Medellín
4       C004     Juan Ríos        Bogotá
5       C005    Sofía Mora  Barranquilla
6       C006    Diana Díaz        Bogotá
7       C007      Ana Soto      Medellín
8       C008  Adriana Cruz          Cali


## **Ejercicio 2: Consolidado trimestral**
Gerencia necesita el crecimiento de cada sucursal entre enero y marzo.

In [56]:
# Usa la tabla consolidado (o crea la tuya con concat).
consolidado

,mes,sucursal,ventas
0,Ene,Norte,117000000.0
1,Ene,Sur,187000000.0
2,Ene,Centro,92000000.0
3,Feb,Norte,152000000.0
4,Feb,Sur,89000000.0
5,Feb,Centro,155000000.0
6,Mar,Norte,85000000.0
7,Mar,Sur,159000000.0
8,Mar,Centro,144000000.0


In [57]:
#Pásala a formato ancho con pivot_table (sucursales × meses).
ancho2 = consolidado.pivot_table(index="sucursal", columns="mes", values="ventas")
ancho2 = ancho2[["Ene", "Feb", "Mar"]]   # meses en orden cronológico
ancho2

mes,Ene,Feb,Mar
sucursal,,,
Centro,92000000.0,155000000.0,144000000.0
Norte,117000000.0,152000000.0,85000000.0
Sur,187000000.0,89000000.0,159000000.0


In [58]:
# Calcula la columna crecimiento_% = (Mar − Ene) / Ene × 100.
# crecimiento_% = (Mar - Ene) / Ene * 100, redondeado a 1 decimal
ancho2["crecimiento_%"] = ((ancho2["Mar"] - ancho2["Ene"]) / ancho2["Ene"] * 100).round(1)
ancho2

mes,Ene,Feb,Mar,crecimiento_%
sucursal,,,,
Centro,92000000.0,155000000.0,144000000.0,56.5
Norte,117000000.0,152000000.0,85000000.0,-27.4
Sur,187000000.0,89000000.0,159000000.0,-15.0


In [63]:
# Ordena de mayor a menor e indica la sucursal líder.
ranking = ancho2.sort_values("crecimiento_%", ascending=False)
print(ranking[["Ene", "Mar", "crecimiento_%"]])

# La primera fila del ranking es la sucursal líder
print("\nSucursal líder:", ranking.index[0], "con", ranking["crecimiento_%"].iloc[0], "%")

mes               Ene          Mar  crecimiento_%
sucursal                                         
Centro     92000000.0  144000000.0           56.5
Sur       187000000.0  159000000.0          -15.0
Norte     117000000.0   85000000.0          -27.4

Sucursal líder: Centro con 56.5 %


## **Ejercicio 3: SQL vs pandas**
Responde la misma pregunta de dos formas: ¿cuál es el medio de pago con más ventas?

In [64]:
# Resuélvelo con pandas: groupby("pago.medio").
ventas_pd = pedidos.groupby("pago.medio")["pago.valor"].sum()
mejor_pd = ventas_pd.sort_values(ascending=False).head(1)
print(mejor_pd)

pago.medio
Tarjeta    360000
Name: pago.valor, dtype: int64


In [66]:
# Crea la tabla pagos en SQLite con to_sql
pagos = pedidos.rename(columns={"pago.medio": "medio", "pago.valor": "valor"})[
    ["pedido", "medio", "valor"]]
pagos.to_sql("pagos", con, index=False, if_exists="replace")
pagos

,pedido,medio,valor
0,101,Tarjeta,150000
1,102,PSE,80000
2,103,Efectivo,45000
3,104,Tarjeta,210000


In [67]:
# Escribe la consulta SELECT ... GROUP BY ... ORDER BY
query = """
SELECT medio, SUM(valor) AS total
FROM pagos
GROUP BY medio
ORDER BY total DESC
LIMIT 1
"""
mejor_sql = pd.read_sql(query, con)
print(mejor_sql)

     medio   total
0  Tarjeta  360000


In [69]:
# Compara: ¿dan el mismo resultado?
print("pandas ->", mejor_pd.index[0], mejor_pd.iloc[0])
print("SQL    ->", mejor_sql["medio"][0], mejor_sql["total"][0])


pandas -> Tarjeta 360000
SQL    -> Tarjeta 360000
